# DehazeFormer Real-Time Dehazing Server

This notebook runs the DehazeFormer AI model on Google Colab's GPU and exposes it as an HTTP API via ngrok.

**How it works:**
1. Loads the DehazeFormer model with pretrained weights on GPU
2. Starts a Flask HTTP server on port 5000
3. Exposes the server to the internet via ngrok tunnel
4. Your Node.js backend sends frames here for dehazing

**Setup steps:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Upload your `dehazeformer_real_haze_best.pth` weights when prompted
4. Copy the ngrok URL and paste it as `COLAB_URL` in your backend `.env`

---

## Step 1: Install Dependencies & Clone DehazeFormer

In [ ]:
# Install required packages
!pip install -q flask pyngrok timm einops opencv-python-headless

# Clone DehazeFormer model architecture
import os
if not os.path.exists('DehazeFormer'):
    !git clone https://github.com/IDKiro/DehazeFormer.git
    print('DehazeFormer cloned successfully')
else:
    print('DehazeFormer already exists')

import sys
sys.path.insert(0, 'DehazeFormer')

# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## Step 2: Configure ngrok Auth Token

**You need a free ngrok account:**
1. Go to [ngrok.com](https://ngrok.com) and sign up (free)
2. Go to [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Copy your auth token and paste it below

In [ ]:
# PASTE YOUR NGROK AUTH TOKEN HERE
NGROK_AUTH_TOKEN = ''  # <-- Paste your token between the quotes

if not NGROK_AUTH_TOKEN:
    print('WARNING: No ngrok token set! Get one free at https://ngrok.com')
    print('Without ngrok, the server will only be accessible within Colab.')
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print('ngrok auth token configured successfully')

## Step 3: Upload Model Weights

Upload your `dehazeformer_real_haze_best.pth` file (54MB).

**Option A:** Run the cell below to upload via browser.

**Option B:** If weights are on Google Drive, mount drive and set the path manually.

In [ ]:
# ==== OPTION A: Upload from your computer ====
WEIGHTS_PATH = '/content/dehazeformer_real_haze_best.pth'

if not os.path.exists(WEIGHTS_PATH):
    print('Upload your dehazeformer_real_haze_best.pth file:')
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        if filename.endswith('.pth'):
            os.rename(filename, WEIGHTS_PATH)
            print(f'Weights saved to {WEIGHTS_PATH}')
            break
else:
    print(f'Weights already exist at {WEIGHTS_PATH}')
    print(f'Size: {os.path.getsize(WEIGHTS_PATH) / 1024 / 1024:.1f} MB')

# ==== OPTION B: Google Drive (uncomment if using Drive) ====
# from google.colab import drive
# drive.mount('/content/drive')
# WEIGHTS_PATH = '/content/drive/MyDrive/your_folder/dehazeformer_real_haze_best.pth'

## Step 4: Load DehazeFormer Model

In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import base64
import time
from models.dehazeformer import DehazeFormer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def load_stage2_model(weights_path):
    """Load DehazeFormer with Stage 2 fine-tuned architecture."""
    model = DehazeFormer(
        in_chans=3, out_chans=4, window_size=8,
        embed_dims=[24, 48, 96, 48, 24],
        mlp_ratios=[2., 4., 4., 2., 2.],
        depths=[12, 12, 12, 6, 6],
        num_heads=[2, 4, 6, 1, 1],
        attn_ratio=[1/4, 1/2, 3/4, 0, 0],
        conv_type=['Conv', 'Conv', 'Conv', 'Conv', 'Conv']
    )

    ckpt = torch.load(weights_path, map_location='cpu')
    state = ckpt.get('state_dict', ckpt.get('model', ckpt))
    new_state = {(k[7:] if k.startswith('module.') else k): v for k, v in state.items()}
    model.load_state_dict(new_state, strict=False)
    print(f'Loaded weights from: {weights_path}')

    return model.to(device).eval()

# Load the model
model = load_stage2_model(WEIGHTS_PATH)
print(f'DehazeFormer loaded on {device}')

# Warm up with a dummy frame to pre-allocate GPU memory
with torch.no_grad():
    dummy = torch.randn(1, 3, 240, 320).to(device)
    _ = model(dummy)
print('Model warmed up and ready for inference')

## Step 5: Start Dehazing Server

This cell starts the Flask server and exposes it via ngrok.

**After running this cell:**
1. Copy the `COLAB_URL` printed below
2. Paste it in your backend's `.env` file as `COLAB_URL=https://xxxx.ngrok-free.app`
3. Restart your backend server

**Keep this notebook running** while using the app!

In [ ]:
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB max request

# Track stats
stats = {'frames_processed': 0, 'total_time': 0, 'errors': 0}

@torch.no_grad()
def dehaze_frame(frame_base64):
    """Dehaze a single base64-encoded frame using DehazeFormer."""
    img_data = base64.b64decode(frame_base64)
    np_arr = np.frombuffer(img_data, np.uint8)
    frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

    if frame is None:
        raise ValueError('Could not decode image from base64')

    h, w, _ = frame.shape
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Pad to multiple of 8 for transformer
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8

    inp = torch.from_numpy(img_rgb / 255.0).float().permute(2, 0, 1).unsqueeze(0).to(device)
    if pad_h > 0 or pad_w > 0:
        inp = F.pad(inp, (0, pad_w, 0, pad_h), mode='reflect')

    # Run inference
    out = model(inp)[:, :3, :h, :w].squeeze(0).permute(1, 2, 0).clamp(0, 1)

    # Convert back
    out_np = (out.cpu().numpy() * 255).astype(np.uint8)
    out_bgr = cv2.cvtColor(out_np, cv2.COLOR_RGB2BGR)

    _, buffer = cv2.imencode('.jpg', out_bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return base64.b64encode(buffer).decode('utf-8')


@app.route('/dehaze', methods=['POST'])
def dehaze_endpoint():
    """Receive a base64 frame, dehaze it, return the result."""
    try:
        data = request.get_json()
        if not data or 'frame' not in data:
            return jsonify({'error': 'Missing "frame" field in request body'}), 400

        start_time = time.time()
        dehazed = dehaze_frame(data['frame'])
        processing_time = (time.time() - start_time) * 1000  # ms

        stats['frames_processed'] += 1
        stats['total_time'] += processing_time

        return jsonify({
            'dehazed': dehazed,
            'processing_time_ms': round(processing_time, 1)
        })

    except Exception as e:
        stats['errors'] += 1
        return jsonify({'error': str(e)}), 500


@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint."""
    avg_time = stats['total_time'] / max(stats['frames_processed'], 1)
    return jsonify({
        'status': 'running',
        'model': 'DehazeFormer Stage 2',
        'device': str(device),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
        'frames_processed': stats['frames_processed'],
        'avg_processing_time_ms': round(avg_time, 1),
        'errors': stats['errors']
    })


# Start Flask in background thread
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, threaded=True),
    daemon=True
).start()

import time as _time
_time.sleep(2)  # Wait for Flask to start

# Expose via ngrok
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(5000)
    colab_url = str(public_url)
    # Ensure we use the clean URL (pyngrok may return NgrokTunnel object)
    if 'NgrokTunnel' in colab_url:
        colab_url = colab_url.split('"')[1]

    print('=' * 60)
    print('DEHAZING SERVER IS RUNNING!')
    print('=' * 60)
    print()
    print(f'  Public URL:  {colab_url}')
    print()
    print('  Add this to your backend .env file:')
    print(f'  COLAB_URL={colab_url}')
    print()
    print('  Health check: ' + colab_url + '/health')
    print('  Dehaze API:   ' + colab_url + '/dehaze (POST)')
    print()
    print('  Keep this notebook running while using the app!')
    print('=' * 60)

except Exception as e:
    print(f'ngrok failed: {e}')
    print('Server is running locally on port 5000 but not accessible externally.')
    print('Make sure you set NGROK_AUTH_TOKEN in Step 2.')

## Step 6: Test the Server (Optional)

Run this cell to test the dehazing endpoint with a sample image.

In [ ]:
import requests
from IPython.display import display, HTML

# Create a simple test image (gray haze-like image)
test_img = np.full((240, 320, 3), 180, dtype=np.uint8)  # Gray foggy image
# Add some variation
test_img[50:190, 80:240] = 120  # Darker region (simulating objects in haze)
_, buf = cv2.imencode('.jpg', test_img)
test_b64 = base64.b64encode(buf).decode('utf-8')

# Test the endpoint
print('Sending test frame to /dehaze...')
response = requests.post('http://localhost:5000/dehaze', json={'frame': test_b64})

if response.status_code == 200:
    result = response.json()
    print(f'Success! Processing time: {result["processing_time_ms"]:.1f}ms')
    print(f'Response size: {len(result["dehazed"])} chars (base64)')

    # Display side-by-side
    display(HTML(f'''
    <div style="display:flex; gap:20px;">
        <div><b>Hazy Input</b><br><img src="data:image/jpeg;base64,{test_b64}" width="320"></div>
        <div><b>Dehazed Output</b><br><img src="data:image/jpeg;base64,{result['dehazed']}" width="320"></div>
    </div>
    '''))
else:
    print(f'Error: {response.status_code} - {response.text}')

# Check health
health = requests.get('http://localhost:5000/health').json()
print(f'\nServer health: {health}')

## Step 7: Monitor Server (Optional)

Run this cell to see real-time stats. It will update every 5 seconds.

In [ ]:
import time
from IPython.display import clear_output

print('Monitoring server stats (interrupt kernel to stop)...')
print()

try:
    while True:
        health = requests.get('http://localhost:5000/health').json()
        clear_output(wait=True)
        print('DehazeFormer Server Monitor')
        print('=' * 40)
        print(f"  Status:      {health['status']}")
        print(f"  Device:      {health['device']}")
        print(f"  GPU:         {health['gpu']}")
        print(f"  Frames:      {health['frames_processed']}")
        print(f"  Avg Time:    {health['avg_processing_time_ms']}ms")
        print(f"  Errors:      {health['errors']}")
        print(f"  Updated:     {time.strftime('%H:%M:%S')}")
        print('=' * 40)
        time.sleep(5)
except KeyboardInterrupt:
    print('Monitoring stopped.')